# TP17 - BlueVisionary

# Iteration 3 - Data Cleaning and Wrangling

In [ ]:
# Install Libraries
#!pip install pandas
#!pip install numpy
#!pip install csv
#!pip install json
#!pip install googlemaps
#!pip install geopy

In [ ]:
# Importing libraries
import pandas as pd
import numpy as np
import csv
import json
import googlemaps
import time
import os
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import warnings

In [ ]:
# Suppress any warning
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

# Load Datasets

In [ ]:
# Get the current working directory (directory of the script)
current_dir = os.getcwd()

# Construct the relative path to the CSV file
processing_facility_1 = os.path.join(current_dir, "147594_01_0.csv")

# Construct the relative path to the CSV file
population_1 = os.path.join(current_dir, "Population capital cities and rest of country, Australia.csv")

# Construct the relative path to the Excel file
national_waste_1 = os.path.join(current_dir, "national-waste-database-2020.xlsx")

In [ ]:
# Reading processing facility data from CSV
process_fac = pd.read_csv(processing_facility_1)

# Reading Population csv
pop_df = pd.read_csv(population_1)

# Reading national waste excel data from 'Database 2020' sheet
national_df = pd.read_excel(national_waste_1, sheet_name='Database 2020')



In [ ]:
process_fac.head()

In [ ]:
pop_df.head()

In [ ]:
national_df.head()

# Processing Facility Dataset

In [ ]:
process_fac.head()

In [ ]:
process_fac.info()

In [ ]:
# Checking nulls values
sum_of_null_values = process_fac.isnull().sum()

# Check for total null values in the entire DataFrame
total_null_values = process_fac.isnull().sum().sum()

print(sum_of_null_values)
print(f"Total null values in DF: {total_null_values}")

In [ ]:
# Checking duplicate rows
duplicate_count = process_fac.duplicated().sum()

# Display rows that are duplicates
duplicate_rows = process_fac[process_fac.duplicated()]

print(f"Total duplicate rows in DF: {duplicate_count}")
print(duplicate_rows)

In [ ]:
# Drop unnecessary columns 
drop_cols = ["GA_ID", "UNIQUE_SITE_ID", "AUTHORITY", "LICENCE_NO", "CO_LOCATED", "SPATIAL_CONFIDENCE", "CAPTURE_METHOD",
"GNAF_ADDRESS_DETAIL_PID", "GNAF_FORMATTED_ADDRESS", "GNAF_SUBURB", "GNAF_POSTCODE", "GNAF_CONFIDENCE", "GNAF_DATE_CREATED", 
"GNAF_DATE_LAST_MODFIED", "GNAF_DATE_RETIRED"]

process_fac.drop(columns = drop_cols, inplace= True)

In [ ]:
# Printing unique values in a column
print(process_fac.FACILITY_INFASTRUCTURE_TYPE.unique())

In [ ]:
# Filtering df based on below values
# 'PLASTICS RECOVERY FACILITY' 
# 'PLASTICS REPROCESSING FACILITY'
process_fac_1 = process_fac.loc[process_fac["FACILITY_INFASTRUCTURE_TYPE"].isin(['PLASTICS RECOVERY FACILITY','PLASTICS REPROCESSING FACILITY'])]


In [ ]:
process_fac_1.info()

In [ ]:
# write the csv for safety purpose
process_fac_1.to_csv('filtered_processing_facility.csv', index=False)

In [ ]:
# assigning to new df
filtered_process_fac= process_fac_1

In [ ]:
filtered_process_fac.head()

In [ ]:
# Changing column values in Proper case 
columns_to_change = ['FACILITY_MANAGEMENT_TYPE','FACILITY_INFASTRUCTURE_TYPE','FACILITY_OWNER','FACILITY_NAME','ADDRESS','SUBURB','OPERATIONAL_STATUS']
filtered_process_fac[columns_to_change] = filtered_process_fac[columns_to_change].apply(lambda x: x.str.title())


In [ ]:
# Changing the datatype of postcode 
filtered_process_fac['POSTCODE'] = filtered_process_fac['POSTCODE'].astype(int).astype(str)

In [ ]:
# Creating a new columns
filtered_process_fac['FULL_ADDRESS'] = filtered_process_fac['ADDRESS'] + ', ' + filtered_process_fac['SUBURB'] + ', ' + filtered_process_fac['STATE'] + ' ' + filtered_process_fac['POSTCODE']



In [ ]:
filtered_process_fac.head()

In [ ]:
# Finding out longitude and latitude 

# Initialize geocoder
geolocator = Nominatim(user_agent="process_fac")

# Create a RateLimiter to avoid overwhelming the service
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

# Function to get latitude and longitude
def get_latitude_longitude(address):
    try:
        location = geocode(address)
        if location:
            return location.latitude, location.longitude
        else:
            return None, None
    except Exception as e:
        return None, None

# Apply the function to get latitude and longitude for each address
filtered_process_fac['LATITUDE'], filtered_process_fac['LONGITUDE'] = zip(*filtered_process_fac['FULL_ADDRESS'].apply(get_latitude_longitude))

In [ ]:
filtered_process_fac.head()

In [ ]:
# Find out number and email addresses

# Initialize the Google Maps API client with your API key
API_KEY = 'YOUR GOOGLE API TOKEN'
gmaps = googlemaps.Client(key=API_KEY)

# Function to get contact details such as contact number and website using facility name
def get_contact_details(facility_name):
    try:
        # Use Places API to search for the place based on the facility name
        places_result = gmaps.places(query=facility_name)

        if places_result['status'] == 'OK':
            place_id = places_result['results'][0]['place_id']
            # Get detailed information about the place
            place_details = gmaps.place(place_id=place_id)
            
            # Extract the website if available
            website = place_details['result'].get('website', "No Website Available")
            # Extract the phone number if available
            phone_number = place_details['result'].get('formatted_phone_number', "No Phone Number Available")

            return phone_number, website
        else:
            return "No Results", "No Results"
    except Exception as e:
        return f"Error: {e}", f"Error: {e}"


df = pd.DataFrame(data)

# Looping through each facility name and get phone number and website
contact_numbers = []
website_links = []

for each_facility in filtered_process_fac['FACILITY_NAME']:
    phone, website = get_contact_details(each_facility)
    contact_numbers.append(phone)
    website_links.append(website)
    time.sleep(1)  # Adding a 1-second delay to avoid rate-limiting

# Add the phone number and website information to the DataFrame
filtered_process_fac['PHONE_NUMBER'] = phone_numbers
filtered_process_fac['WEBSITE'] = websites


In [ ]:
filtered_process_fac.head()

In [ ]:
# Arranging the columns
filtered_process_fac = filtered_process_fac[['OBJECTID','UNIQUE_RECORD_ID',
                              'FACILITY_MANAGEMENT_TYPE',
                              'FACILITY_INFASTRUCTURE_TYPE',
                              'FACILITY_OWNER',
                              'FACILITY_NAME',
                              'ADDRESS',
                              'SUBURB',
                              'POSTCODE',
                              'OPERATIONAL_STATUS', 
                              'FULL_ADDRESS',
                              'LATITUDE',
                              'LONGITUDE',
                              'PHONE_NUMBER','WEBSITE']]

In [ ]:
filtered_process_fac.shape

In [ ]:
filtered_process_fac.to_csv('processing_facilities_contact.csv', index=False)

# National Waste Dataset

In [ ]:
# Alternate way to load excel data from 'Database 2020' sheet
# national_df = pd.read_excel("national-waste-database-2020.xlsx", sheet_name='Database 2020')

In [ ]:
national_df.head()

In [ ]:
national_df.info()

In [ ]:
# Dropping unecessary columns
drop_cols_nat = ["Type", "Classification", "Total type", "Sub-stream", "Type no."]
national_df.drop(columns = drop_cols_nat, inplace = True)

In [ ]:
# proper case in "Jurisdiction"
national_df['Jurisdiction'] = national_df['Jurisdiction'].str.upper()

In [ ]:
# filtering data based on "Category" == "Plastics"
national_df_cat = national_df.loc[national_df["Category"].isin(['Plastics'])]


In [ ]:
national_df_cat.head()

In [ ]:
# filter data based on "Stream" == "MSW" (Municipality waste == household waste)
national_df_cat_st = national_df_cat.loc[national_df_cat["Stream"].isin(["MSW"])]


In [ ]:
national_df_cat_st.head()

In [ ]:
# Checking duplicate rows
duplicate_count_national_df = national_df_cat_st.duplicated().sum()

# Display rows that are duplicates
duplicate_rows_national_df = national_df_cat_st[national_df_cat_stc.duplicated()]

print(f"Total duplicate rows in DF: {duplicate_count_national_df}")
print(duplicate_rows_national_df)

In [ ]:
national_df_cat_st.shape

In [ ]:
# Checking nulls
national_df_cat_st.isnull().sum()

In [ ]:
# aggregating data based on years and states
aggregated_df_year_state = national_df_cat_st.groupby(['Year', 'Jurisdiction']).agg({'Tonnes': 'sum'}).reset_index()
aggregated_df_year_state.head(50)

In [ ]:
# aggregating data based on years and states and Management
aggregated_df_year_state_man = national_df_cat_st.groupby(['Year', 'Jurisdiction', 'Management']).agg({'Tonnes': 'sum'}).reset_index()
aggregated_df_year_state_man.head(50)

In [ ]:
# Filtering data for Recycling
aggregated_df_year_state_man = aggregated_df_year_state_man.loc[aggregated_df_year_state_man["Management"] == "Recycling"]

In [ ]:
aggregated_df_year_state_man.head()

In [ ]:
# Creating columns for Kilograms and Grams
aggregated_df_year_state_man["Kilograms"] = aggregated_df_year_state_man["Tonnes"]*1000
aggregated_df_year_state_man["Grams"] = aggregated_df_year_state_man["Kilograms"]*1000

In [ ]:
aggregated_df_year_state_man

In [ ]:
aggregated_df_year_state_man_nostate = aggregated_df_year_state_man.groupby(['Year', 'Management']).agg({'Tonnes': 'sum', 'Kilograms':'sum', 'Grams':'sum'}).reset_index()

In [ ]:
aggregated_df_year_state_man_nostate.head()

In [ ]:
# not necessary to save this data to csv 
filtered_process_fac.to_csv('national_cleaned_data.csv', index=False)

# Population Dataset

In [ ]:
# Alternate way to load population csv
# pop_df = pd.read_csv("Population capital cities and rest of country, Australia.csv")

In [ ]:
pop_df.head()

In [ ]:
pop_df.info()

In [ ]:
# Remove commas and convert columns to integer type
pop_df['Total Capital Cities (pop.)'] = pop_df['Total Capital Cities (pop.)'].str.replace(',', '').astype(int)
pop_df['Rest of States/Territories (pop.)'] = pop_df['Rest of States/Territories (pop.)'].str.replace(',', '').astype(int)


In [ ]:
# Filtering population based on years
pop_df= pop_df.loc[(pop_df['Year']>= 2006) & (pop_df['Year']<= 2019)].reset_index(drop=True)

In [ ]:
pop_df.head(150)

In [ ]:
# Creating bins from 2006-2007 to 2018-2019
bins=[2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2020]
bin_labels=[f"{year}-{year+1}" for year in range(2006, 2019)]


pop_df['Year Bin'] = pd.cut(pop_df['Year'], bins=bins, labels=bin_labels, right=False)

# Dropping the unnecessary 'Year' column to match your final requirement
df_pop_final = pop_df[['Year Bin', 'Total Capital Cities (pop.)', 'Rest of States/Territories (pop.)']]


In [ ]:
df_pop_final.head()

In [ ]:
# Creating columns
df_pop_final["Total Population"] = df_pop_final["Total Capital Cities (pop.)"] + df_pop_final["Rest of States/Territories (pop.)"]

In [ ]:
# Finalising population dataset
df_pop_final = df_pop_final[['Year Bin', 'Total Population']]

# Renamming the column
df_pop_final = df_pop_final.rename(columns={"Year Bin": "Year"})

In [ ]:
df_pop_final.head()

In [ ]:
# Contribution per person per month 

# Merging dataframes
pop_merged = pd.merge(aggregated_df_year_state_man_nostate, df_pop_final, how='left', on=['Year'])

In [ ]:
pop_merged.head()

In [ ]:
# contributuion per month in one year

pop_merged["Contribution per month(Tonnes)"] = pop_merged["Tonnes"]/(pop_merged['Total Population']*12)
pop_merged["Contribution per month(Kg)"] = pop_merged["Kilograms"]/(pop_merged['Total Population']*12)
pop_merged["Contribution per month(Grams)"] = pop_merged["Grams"]/(pop_merged['Total Population']*12)

In [ ]:
pop_merged.head()

In [ ]:
pop_merged[["Tonnes", "Kilograms", "Grams","Total Population",
    "Contribution per month(Tonnes)",
    "Contribution per month(Kg)",
    "Contribution per month(Grams)"]] = pop_merged[["Tonnes", "Kilograms", "Grams",
                                                      "Total Population",
                                                      "Contribution per month(Tonnes)",
                                                      "Contribution per month(Kg)",
                                                      "Contribution per month(Grams)"]].round(4)

In [ ]:
pop_merged.to_csv('individual_contribution_permonth.csv', index=False)

In [ ]:
# Overall contribution
aggregated_contribution = pop_merged.groupby(['Management']).agg({'Tonnes':'mean', 'Kilograms': 'mean', 'Grams':'mean',
                                                                  'Total Population':'mean',
                                                                  'Contribution per month(Tonnes)': 'mean',
                                                                  'Contribution per month(Kg)': 'mean', 
                                                                  'Contribution per month(Grams)': 'mean' }).reset_index()

In [ ]:
# Rounding off values upto 4 decimal places in multiple columns 
aggregated_contribution[["Tonnes", "Kilograms", "Grams","Total Population",
    "Contribution per month(Tonnes)",
    "Contribution per month(Kg)",
    "Contribution per month(Grams)"]] = aggregated_contribution[["Tonnes", "Kilograms", "Grams",
                                                      "Total Population",
                                                      "Contribution per month(Tonnes)",
                                                      "Contribution per month(Kg)",
                                                      "Contribution per month(Grams)"]].round(4)

In [ ]:
aggregated_contribution.head()

In [ ]:
# Writing data to csv
aggregated_contribution.to_csv('average_individual_contribution_permonth.csv', index=False)